In [1]:
import pandas as pd
import os

In [2]:
DATA_DIR = '/home/mkeoliya/projects/CAREBench/data/mimiciv/3.1'

In [3]:
COLS = {
    'chartevents': ('subject_id', 'hadm_id', 'stay_id', 'charttime', 'itemid', 'valuenum', 'value'),
    'inputevents': ('subject_id', 'hadm_id', 'stay_id', 'starttime', 'itemid', 'amount', 'endtime', 'patientweight'),
    'outputevents': ('subject_id', 'hadm_id', 'stay_id', 'charttime', 'itemid', 'value'),
    'procedureevents': ('subject_id', 'hadm_id', 'stay_id', 'starttime', 'itemid', 'value', 'endtime', 'patientweight'),
    'ingredientevents': ('subject_id', 'hadm_id', 'stay_id', 'starttime', 'endtime', 'itemid', 'amount'),
    'labevents': ('subject_id', 'hadm_id', 'charttime', 'itemid', 'valuenum', 'value', 'ref_range_lower', 'ref_range_upper'),
    'pharmacy': ('subject_id', 'hadm_id', 'starttime', 'medication'),
    'emar': ('subject_id', 'hadm_id', 'charttime', 'medication', 'event_txt'),
    'microbiologyevents': ('subject_id', 'hadm_id', 'charttime', 'test_name', 'spec_itemid', 'test_itemid'),
    'datetimeevents': ('subject_id', 'hadm_id', 'stay_id', 'itemid', 'value'),
    }


MORTALITY LABEL

In [150]:
df = pd.read_csv(os.path.join(DATA_DIR, 'hosp', 'admissions.csv'), usecols=['subject_id', 'hadm_id', 'deathtime', 'admittime', 'dischtime'])

In [152]:
df.loc[:, 'deathtime'] = pd.to_datetime(df['deathtime'])
df.loc[:, 'admittime'] = pd.to_datetime(df['admittime'])
df.loc[:, 'dischtime'] = pd.to_datetime(df['dischtime'])

In [ ]:
pos_df = df[~df['deathtime'].isna()]
max_intime = pos_df.groupby('subject_id')['admittime'].transform('max')
pos_df = pos_df[pos_df['admittime'] == max_intime]

In [174]:
stay_df = pd.read_csv(os.path.join(DATA_DIR, 'icu', 'icustays.csv'), usecols=['hadm_id'])

In [175]:
neg_df = df[df['deathtime'].isna()]
neg_df = neg_df.drop(columns='deathtime')
neg_df = neg_df[~neg_df['subject_id'].isin(pos_df['subject_id'])]
neg_df = neg_df[neg_df['hadm_id'].isin(stay_df['hadm_id'])]

In [178]:
label_df = pd.concat([pos_df, neg_df], ignore_index=True)

In [180]:
label_df.to_csv(os.path.join(DATA_DIR, 'label_mortality.csv'), index=False)

DEMOGRAPHICS

In [ ]:
adm_df = pd.read_csv(os.path.join(DATA_DIR, 'hosp', 'admissions.csv'))
pat_df = pd.read_csv(os.path.join(DATA_DIR, 'hosp', 'patients.csv'))

In [ ]:
pat_df = pat_df.merge(adm_df[['subject_id', 'race']], on='subject_id', how='right')
pat_df = pat_df.drop(columns='dod')
adm_df = adm_df.drop(columns=['race'])

In [ ]:
adm_df = adm_df.rename(columns={f'{DATA_DIR}/subject_id': 'subject_id'})
adm_df = adm_df.drop(columns=['hospital_expire_flag', 'admission_type', 'admit_provider_id', 'admission_location', 'discharge_location', 'insurance', 'language', 'marital_status'])
adm_df.to_csv(os.path.join(DATA_DIR, 'hosp', 'admissions.csv'), index=False)

In [ ]:
import numpy as np

In [39]:
def race_str(x):
    if 'ASIAN' in x:
        return 'RACE/ASIAN'
    elif 'BLACK' in x:
        return 'RACE/BLACK'
    elif 'WHITE' in x:
        return 'RACE/WHITE'
    elif 'HISPANIC' in x:
        return 'RACE/HISPANIC'
    elif 'BLACK' in x:
        return 'RACE/BLACK'
    elif 'NATIVE' in x:
        return 'RACE/AMERICAN'
    elif x == 'PORTUGUESE': return 'RACE/PORTUGUESE'
    elif x == 'SOUTH AMERICAN': return 'RACE/SOUTH'
    elif x in ['MULTIPLE RACE/ETHNICITY', 'OTHER', 'UNABLE TO OBTAIN', 'UNKNOWN', 'PATIENT DECLINED TO ANSWER', 'MULTIPLE RACE/ETHNICITY']:
        return 'RACE/UNKNOWN'

In [40]:
AGES = [0, 20, 30, 40, 50, 60, 70, 80, 90, 100]
AGE_LABELS = [f'Age/{AGES[i]}-{AGES[i+1]}' for i in range(0, len(AGES)-1)]
def age_str(x):
    ind = np.searchsorted(AGES, x, 'right')
    return AGE_LABELS[ind-1]

In [ ]:
pat_df['Gender_str'] = 'Gender/' + pat_df['gender']
pat_df['Race_str'] = pat_df['race'].apply(race_str)
pat_df['Age_str'] = pat_df['anchor_age'].apply(age_str)

In [52]:
pat_df['Gender_ind'] = pd.factorize(pat_df['gender'])[0]
pat_df['Race_ind'] = pd.factorize(pat_df['Race_str'])[0]
pat_df['Age_ind'] = pat_df['anchor_age']

In [54]:
pat_df = pat_df.rename(columns={'anchor_age': 'age'})
pat_df = pat_df.drop(columns=['anchor_year', 'anchor_year_group'])

In [56]:
pat_df.to_csv(os.path.join(DATA_DIR, 'hosp', 'patients.csv'), index=False)

BUCKETS

In [3]:
import numpy as np

In [4]:
scale = 10_000

In [ ]:
def valuenum(DIV, FNAME, VCOL, MCOL='itemid'):
    print(f'Running: {FNAME}')
    df = pd.read_csv(os.path.join(DATA_DIR, f'{DIV}', f'{FNAME}.csv'), usecols=[f'{MCOL}', f'{VCOL}'])
    df = df.dropna()

    df_dict = {}
    for c in df[f'{MCOL}'].unique():
        df_c = df[df[f'{MCOL}'] == c]
        if df_c[f'{VCOL}'].nunique() <= 1:
            df_dict[c] = [c]
        else:
            bucket = np.percentile(df_c[f'{VCOL}'], np.arange(0, 110, 10), method='lower')
            bucket = bucket[np.insert(np.diff(bucket) != 0, 0, True)]
            bucket = np.array([
                    np.floor(bucket[0] * scale) / scale,             # floor first
                    *[round(b, ndigits=4) for b in bucket[1:-1]],    # round middle
                    np.ceil(bucket[-1] * scale) / scale              # ceil last
                ])
            df_dict[c] = bucket[np.concatenate(([True], bucket[1:] != bucket[:-1]))]

    with open(os.path.join(DATA_DIR, f'{FNAME}_buckets.pkl'), 'wb') as f:
        pickle.dump(df_dict, f)

In [ ]:
valuenum('hosp', 'labevents', 'valuenum')
valuenum('icu', 'chartevents', 'valuenum')
valuenum('icu', 'ingredientevents', 'amount')
valuenum('icu', 'inputevents', 'amount')
valuenum('icu', 'outputevents', 'value')
valuenum('icu', 'procedureevents', 'value')

EVENTVAL

In [7]:
import pickle
import numpy as np

In [8]:
def bucket_eventval(event, val, d):
    # event = int(event)
    if event in d.keys():
        buckets = d[event]
        if type(buckets) == str: 
            return buckets
        ind = np.searchsorted(buckets, val, side='right')
        if ind == len(buckets):
            eventval = f"{event}|{buckets[ind-1]}-"
        else:
            eventval = f"{event}|{buckets[ind-1]}-{buckets[ind]}"
        return eventval
    else:
        print(event)
        return f'{event}'

In [9]:
def bucket_ind(event, val, d):
    buckets = d[event]
    if type(buckets) == str: 
        return 0
    ind = np.searchsorted(buckets, val, side='right')
    return ind

In [10]:
def add_eventval(SPLIT, FNAME, VCOL, MCOL='itemid'):
    with open(os.path.join(DATA_DIR, f'{FNAME}_buckets.pkl'), 'rb') as file:
        buckets = pickle.load(file)

    df = pd.read_csv(os.path.join(DATA_DIR, f'{SPLIT}', f'{FNAME}.csv'))
    df = df[list(COLS[f'{FNAME}'])]
    df['eventval'] = df.apply(lambda x: None if np.isnan(x[f'{VCOL}']) else bucket_eventval(x[f'{MCOL}'], x[f'{VCOL}'], buckets), axis=1)
    df['bucket'] = df.apply(lambda x: None if np.isnan(x[f'{VCOL}']) else bucket_ind(x[f'{MCOL}'], x[f'{VCOL}'], buckets), axis=1)

    df.to_csv(os.path.join(DATA_DIR, f'{SPLIT}', f'{FNAME}.csv'), index=False)

In [ ]:
add_eventval('icu', 'chartevents', 'valuenum')
add_eventval('hosp', 'labevents', 'valuenum')
add_eventval('icu', 'ingredientevents', 'amount')
add_eventval('icu', 'outputevents', 'value')
add_eventval('icu', 'procedureevents', 'value',)
add_eventval('icu', 'inputevents', 'amount')

In [ ]:
lab_df = pd.read_csv(os.path.join(DATA_DIR, 'hosp', 'labevents.csv'))
lab_df = lab_df.rename(columns={'charttime': 'Time', 'itemid': 'Measure', 
    'valuenum': 'Value', 'ref_range_lower': 'Low', 'ref_range_upper': 'High', 'value': 'str_val'})
lab_df.to_csv(os.path.join(DATA_DIR, 'hosp', 'labevents.csv'), index=False)

In [ ]:
proc_df = pd.read_csv(os.path.join(DATA_DIR, 'icu', 'procedureevents.csv'))
proc_df = proc_df.rename(columns={'starttime': 'Time', 'itemid': 'Measure', 'endtime': 'Result_time', 'value': 'Value'})
proc_df['eventval'] = proc_df['eventval'].apply(lambda x: x.replace("[", "").replace("]", ""))
proc_df.to_csv(os.path.join(DATA_DIR, 'icu', 'procedureevents.csv'), index=False)

In [ ]:
input_df = pd.read_csv(os.path.join(DATA_DIR, 'icu', 'inputevents.csv'))
input_df['eventval'] = input_df['eventval'].apply(lambda x: x.replace("[", "").replace("]", ""))
input_df = input_df.rename(columns={'starttime': 'Time', 'itemid': 'Measure', 'endtime': 'Result_time', 'amount': 'Value'})
input_df.to_csv(os.path.join(DATA_DIR, 'icu', 'inputevents.csv'), index=False)

In [ ]:
output_df = pd.read_csv(os.path.join(DATA_DIR, 'icu', 'outputevents.csv'))
output_df = output_df.rename(columns={'charttime': 'Time', 'itemid': 'Measure', 'value': 'Value'})
output_df.to_csv(os.path.join(DATA_DIR, 'icu', 'outputevents.csv'), index=False)

In [ ]:
ingredient_df = pd.read_csv(os.path.join(DATA_DIR, 'icu', 'ingredientevents.csv'))
ingredient_df = ingredient_df.rename(columns={'starttime': 'Time', 'endtime': 'Result_time', 'itemid': 'Measure', 'amount': 'Value'})
ingredient_df.to_csv(os.path.join(DATA_DIR, 'icu', 'ingredientevents.csv'), index=False)

In [ ]:
chart_df = pd.read_csv(os.path.join(DATA_DIR, 'icu', 'chartevents.csv'))
chart_df['eventval'] = chart_df['eventval'].apply(lambda x: x.replace("[", "").replace("]", "") if type(x) == str else x)
chart_df = chart_df.rename(columns={'charttime': 'Time', 'itemid': 'Measure', 'valuenum': 'Value', 'value': 'str_val'})
chart_df.to_csv(os.path.join(DATA_DIR, 'icu', 'chartevents.csv'), index=False)

EVENTVAL: MED

In [ ]:
emar_df = pd.read_csv(os.path.join(DATA_DIR, 'hosp', 'emar.csv'))
micro_df = pd.read_csv(os.path.join(DATA_DIR, 'hosp', 'microbiologyevents.csv'))
date_df = pd.read_csv(os.path.join(DATA_DIR, 'icu', 'datetimeevents.csv'))

In [ ]:
emar_df = emar_df[list(COLS['emar'])]
emar_df = emar_df.rename(columns={'charttime': 'Time', 'medication': 'Measure'})
emar_df['eventval'] = emar_df['Measure']
emar_df['bucket'] = 0


In [ ]:
micro_df = micro_df[list(COLS['microbiologyevents'])]
micro_df['Measure'] = pd.factorize(micro_df[['spec_itemid', 'test_itemid']].apply(tuple, axis=1))[0]
micro_df = micro_df.drop(columns=['spec_itemid', 'test_itemid'])
micro_df = micro_df.rename(columns={'charttime': 'Time'})

In [ ]:
date_df = date_df[list(COLS['datetimeevents'])]
date_df = date_df.rename(columns={'value': 'Time', 'itemid': 'Measure'})
date_df['eventval'] = date_df['Measure']
date_df['bucket'] = 0

In [ ]:
emar_df.to_csv(os.path.join(DATA_DIR, 'hosp', 'emar.csv'), index=False)
micro_df.to_csv(os.path.join(DATA_DIR, 'hosp', 'microbiologyevents.csv'), index=False)
date_df.to_csv(os.path.join(DATA_DIR, 'icu', 'datetimeevents.csv'), index=False)

PIVOT: For classical models?

In [ ]:
def pivot_df(df):
    df_pivoted = df.pivot(index=['patient_id', 'visit_num'], columns='Event')
    df_pivoted.columns = [f'{col}_{event}' for col, event in df_pivoted.columns.swaplevel()]
    df_pivoted = df_pivoted.reindex(sorted(df_pivoted.columns), axis=1)
    df_pivoted = df_pivoted.reset_index()

    df_pivoted['patient_id'] = pd.to_numeric(df_pivoted['patient_id'])
    return df_pivoted

In [ ]:
visits_df = pd.read_parquet(os.path.join(DATA_DIR, f'visits_{i}.parquet'))
visits_df = visits_df.sort_values(['subject_id', 'intime'])
visits_df['episode'] = (
    visits_df[visits_df['intime'].notna()]
    .sort_values(['subject_id', 'intime'])
    .groupby('subject_id')
    .cumcount() + 1
)

GCS

In [ ]:
EYE, MOTOR, VERBAL = [], [], []

In [ ]:
def get_gcs_values(split, files):
    for file in files:
        df = pd.read_parquet(os.path.join(DATA_DIR, split, file))
        EYE += list(df[df['Event'] == 'Glascow coma scale eye opening']['Value'].unique())
        EYE = list(set(EYE))

        MOTOR += list(df[df['Event'] == 'Glascow coma scale motor response']['Value'].unique())
        MOTOR = list(set(MOTOR))

        VERBAL += list(df[df['Event'] == 'Glascow coma scale verbal response']['Value'].unique())
        VERBAL = list(set(VERBAL))

In [ ]:
train_files = os.listdir(os.path.join(DATA_DIR, 'train'))
get_gcs_values('train', train_files)

In [ ]:
test_files = os.listdir(os.path.join(DATA_DIR, 'test'))
get_gcs_values('test', test_files)

COMPUTE STATS

In [ ]:
import numpy as np
from scipy.stats import linregress, kurtosis, skew

In [ ]:
ROOT_DIR = 'data/'
TASK = 'in-hospital-mortality'
DATA_DIR = os.path.join(ROOT_DIR, TASK)

In [ ]:
TRAIN_FILES = os.listdir(os.path.join(DATA_DIR, 'train'))
TEST_FILES = os.listdir(os.path.join(DATA_DIR, 'test'))

In [ ]:
def extract_stats(xs):
    xs = np.array(xs, dtype=float)
    x = np.arange(len(xs))
    slope, _, _, _, _ = linregress(x, xs)

    return pd.Series({
        'min': xs.min(),
        'max': xs.max(),
        'mean': xs.mean(),
        'std': xs.std(),
        'median': np.median(xs),
        'skew': skew(xs),
        'kurtosis': kurtosis(xs),
        'slope': slope,
        'qr_25': np.quantile(xs, 0.25),
        'qr_75': np.quantile(xs, 0.75),
    })

In [ ]:
EYE = ['Spontaneously', 'To Speech', 'To Pain'] 
MOTOR = ['No response', 'Localizes Pain', 'Flex-withdraws', 'Abnormal Flexion', 'Obeys Commands', 'Abnormal extension']
VERBAL = ['Incomprehensible sounds', 'No Response-ETT', 'Confused', 'Oriented', 'No Response', 'Inappropriate Words']

def parse_gcs(x):
    if x['Event'] == 'Glascow coma scale eye opening':
        return EYE.index(x['Value'])
    elif x['Event'] == 'Glascow coma scale motor response':
        return MOTOR.index(x['Value'])
    elif x['Event'] == 'Glascow coma scale verbal response':
        return VERBAL.index(x['Value'])
    else:
        return x['Value']

In [ ]:
def compute_stats(split, files):
    for file in files:
        # print(file[-7:])
        if not file[-7:] == 'parquet': 
            continue
        df = pd.read_parquet(os.path.join(DATA_DIR, split, file))
        if not 'Value' in df.columns: continue
        df['Value'] = df.apply(parse_gcs, axis=1)
        df['Value'] = pd.to_numeric(df['Value'])

        df = (df.sort_values('Hours')[['Event', 'Value']]).groupby('Event').agg(list).reset_index()
        df2 = df['Value'].apply(extract_stats).apply(pd.Series)

        df = df.drop(columns=['Value'])
        df = pd.concat([df, df2], axis=1)
        df.to_parquet(os.path.join(DATA_DIR, split, file))

In [ ]:
compute_stats('train', TRAIN_FILES)
compute_stats('test', TEST_FILES)

PLOT

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('data/mortality/mortality_labels.csv')

In [ ]:
# Assuming your dataframe is named df and intime/outtime are datetime types
df['intime'] = pd.to_datetime(df['intime'])
df['outtime'] = pd.to_datetime(df['outtime'])
df['deathtime'] = pd.to_datetime(df['deathtime'])
df['duration'] = (df['outtime'] - df['intime']).dt.total_seconds() / 60 
df['duration2'] = (df['deathtime'] - df['intime']).dt.total_seconds() / 60  # duration in minutes

# Plot
plt.figure(figsize=(10, 6))
for label, group in df.groupby('y_true'):
    if label == 1:
        plt.hist(group['duration2'], bins=50, alpha=0.5, label=f'Label {label}', density=True)
        # plt.hist(group['duration'], bins=50, alpha=0.5, label=f'Label {label}', density=True)
    else:
        plt.hist(group['duration'], bins=50, alpha=0.5, label=f'Label {label}', density=True)

plt.xlabel('Duration (minutes)')
plt.ylabel('Density')
plt.title('Duration Distribution by Label')
plt.legend()
plt.grid(True)
plt.show()
